# Caching pass: models -> prompts -> activations and rating continuations

The GPU half of the arc-length pipeline. For every model in `MODELS` it generates the prompt corpora, caches the final-prompt-token layer outputs the surface is fitted from, caches the activations of every inference corpus, and generates the stated-stakes rating continuations. It fits nothing and computes no coordinates: `notebooks/2_bcpc_out_of_fold.ipynb` (the BCPC and `bcpc_arc_length`) and `notebooks/3_pls_arc_length_surface.ipynb` (the PLS surface) do that from the caches written here, on any machine, with no GPU and no Hugging Face access.

Each model is loaded exactly once and that one instance serves every stage. No module in `scripts/` loads weights on its own, so a stage that is handed no model raises instead of quietly pulling a second copy onto the GPU. After each model the weights are released and that repository is removed from the local Hugging Face download cache.

The cached layer is selected independently for every model as `floor(0.6 * num_hidden_layers)`. The settings the analysis half must reuse are recorded in `artifacts/<model name>/run_config.json`, so that half never has to be told which layer or batch size a cache was made with.

In [ ]:
from collections import Counter
from pathlib import Path
import gc
import math
import sys
import traceback

import torch
from huggingface_hub import scan_cache_dir
from IPython.display import display
from transformers import AutoConfig

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts' / 'pipeline_config.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts import (cache_activations, inference_datasets, prompt_datasets,
                     stated_stakes)
from scripts.pipeline_config import (NO_TIME_CORPORA, RunConfig,
                                     naming_convention_spec, save_run_config)

print('Repository:', ROOT)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU)')
print(f'{len(inference_datasets.DATASETS)} inference corpora:',
      ', '.join(dataset.name for dataset in inference_datasets.DATASETS))

## Configuration

Add `(model_name, naming_convention)` pairs to `MODELS` in execution order. Use `"llama"` for models whose decoder blocks are named `model.layers.N`, `"gemma4"` for Gemma 4 multimodal checkpoints, and `"mistral3"` for Mistral 3 multimodal checkpoints (its loader also installs the chat template from `chat_template.json` and sends an explicit empty system message, so the template does not inject its dated default system prompt). To support another convention, add its dotted config layer-count attribute and module-path template to `NAMING_CONVENTIONS` in `scripts/pipeline_config.py`; add a loader branch in `cache_activations.load_model` only if the architecture cannot use the standard causal-LM loader. Cache keys remain `layer_out/N` for downstream compatibility. For gated models, authenticate before starting. `BATCH_SIZE` applies to every model; lower it if needed. Compatible caches are reused unless `FORCE` is enabled.

`HF_CACHE_DIR` is the directory Hugging Face downloads model weights into. Leave it `None` to keep Hugging Face's own default (`HF_HOME/hub`, or `HF_HUB_CACHE` when that is set); give it a path to put the weights on a larger or faster volume, which matters on the rented GPU hosts this half tends to run on. The same directory is used for loading and for the cleanup scan, so a custom location is still reclaimed.

`RATING_DATASETS` selects which inference corpora are additionally rated by the model itself; set it to `[]` to skip that pass, which is the only stage that generates text rather than reading activations. The analysis half exports whichever ratings it finds, so skipping the pass here simply leaves those CSVs unwritten.

In [ ]:
MODELS = [
    ('Qwen/Qwen3-4B-Instruct-2507', 'llama'),
    ('Qwen/Qwen3-8B', 'llama'),
    ('Qwen/Qwen3-14B', 'llama'),
    ('mistralai/Mistral-Small-3.1-24B-Instruct-2503', 'mistral3'),
    ('Qwen/Qwen3-32B', 'llama'),
    ('google/gemma-4-31B-it', 'gemma4'),
]
BATCH_SIZE = 256
STAKES_MERGES = {
    'near_existential': 'existential',
    'medium_low': 'medium',
}
POSITION = -1                    # final prompt token
CORPORA = list(NO_TIME_CORPORA)  # quick smoke run: ['conversational_no_time']
FORCE = False                    # rebuild caches that fail fingerprint checks
HF_CACHE_DIR = None              # where model weights are downloaded; None = Hugging Face default
# Corpora whose prompts are also rated by the model itself; [] skips the pass.
RATING_DATASETS = list(stated_stakes.DATASETS)
RATING_TOKENS = stated_stakes.GENERATED_TOKENS  # tokens generated per rating

## Pipeline helpers

`cache_model` loads the model once and reuses that instance for the fitting caches, for every inference corpus, and for the rating continuations, each of which would otherwise load its own copy. The model is released before the call returns, so only one is ever loaded at a time.

`report_placement` prints the accelerate `device_map="auto"` placement that `cache_activations.load_model` produces, and warns when accelerate has offloaded shards to `cpu` or `disk`, which makes every forward pass of every stage markedly slower.

Cache cleanup is repository-scoped: it removes all cached revisions of the completed model ID while allowing Hugging Face to preserve blobs shared by other repositories. It never targets this repository's `artifacts/` tree.

In [ ]:
def cached_layer_for_model(model_name, naming_convention):
    count_attribute, _ = naming_convention_spec(naming_convention)
    architecture = AutoConfig.from_pretrained(
        model_name, trust_remote_code=True,
        cache_dir=None if HF_CACHE_DIR is None else str(HF_CACHE_DIR))
    layer_config = architecture
    for attribute in count_attribute.split('.'):
        layer_config = getattr(layer_config, attribute, None)
        if layer_config is None:
            break
    num_layers = layer_config
    if not isinstance(num_layers, int) or num_layers < 1:
        raise ValueError(f'{model_name}: config has no positive integer {count_attribute}')
    layer = math.floor(0.6 * num_layers)
    if not 0 <= layer < num_layers:
        raise ValueError(f'{model_name}: computed invalid layer {layer} for {num_layers} layers')
    return num_layers, layer


def report_placement(model):
    """Summarise an accelerate `device_map='auto'` placement and flag CPU/disk offload."""
    placement = getattr(model, 'hf_device_map', None)
    if not placement:
        print(f'Placement: single device, {model.device}')
        return False
    counts = Counter(str(device) for device in placement.values())
    print('Placement:', ', '.join(f'{device}: {n} modules' for device, n in sorted(counts.items())))
    offloaded = sorted(device for device in counts if device in ('cpu', 'disk'))
    if offloaded:
        print(f'WARNING: {offloaded} offload in use. Every forward pass moves those shards, '
              f'so all caching stages run markedly slower. Reduce BATCH_SIZE or CORPORA, '
              f'or use a larger GPU.')
    return bool(offloaded)


def delete_local_model_data(model_name):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if HF_CACHE_DIR is not None and not Path(HF_CACHE_DIR).is_dir():
        print(f'No Hugging Face cache directory at {HF_CACHE_DIR}; nothing to delete.')
        return
    cache = scan_cache_dir(HF_CACHE_DIR)
    revisions = [revision.commit_hash
                 for repo in cache.repos
                 if repo.repo_type == 'model' and repo.repo_id == model_name
                 for revision in repo.revisions]
    if not revisions:
        print(f'No local Hugging Face model cache remains for {model_name}.')
        return
    strategy = cache.delete_revisions(*revisions)
    print(f'Deleting {model_name} download cache ({strategy.expected_freed_size_str}) ...')
    strategy.execute()


def cache_model(model_name, naming_convention):
    """Cache everything the analysis half needs from one model, on one load."""
    num_layers, layer = cached_layer_for_model(model_name, naming_convention)
    config = RunConfig(model_name=model_name, naming_convention=naming_convention,
                       layer_component=f'layer_out/{layer}',
                       position=POSITION, batch_size=BATCH_SIZE, stakes_merges=STAKES_MERGES,
                       hf_cache_dir=HF_CACHE_DIR)
    print()
    print(f'=== {model_name}: caching layer floor(0.6 * {num_layers}) = {layer} ===')
    print(config.describe())
    print(f'Cached module: {config.layer_module_name} ({naming_convention})')
    print('Run settings recorded in', save_run_config(config, force=FORCE))

    model = tokenizer = None
    try:
        datasets = prompt_datasets.generate_datasets(config, CORPORA)
        prompt_datasets.preflight(datasets)
        model, tokenizer = cache_activations.load_model(config)
        print(f'Loaded {model_name} on {model.device}')
        report_placement(model)

        cache_paths = cache_activations.run(config, datasets, model=model, tokenizer=tokenizer,
                                            force=FORCE)
        print(f'{len(cache_paths)} fitting caches in {config.activations_dir}')

        for dataset in inference_datasets.DATASETS:
            paths = dataset.cache(config, model, tokenizer, force=FORCE)
            print(f'{dataset.label}: {len(paths)} caches in {dataset.directory(config)}')

        for name, paths in stated_stakes.cache_all(config, RATING_DATASETS, model=model,
                                                   tokenizer=tokenizer, force=FORCE,
                                                   max_new_tokens=RATING_TOKENS).items():
            print(f'{name} ratings: {len(paths)} generation caches')
    finally:
        del model, tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return config.run_dir

## Cache every model sequentially

A failed model does not prevent later entries from running. Its traceback is recorded, local model data is still cleaned up, and the cell raises after printing the complete status table.

In [ ]:
if not MODELS:
    raise ValueError('MODELS must contain at least one (model_name, naming_convention) pair.')
for entry in MODELS:
    if not isinstance(entry, (tuple, list)) or len(entry) != 2 or not all(isinstance(value, str) and value for value in entry):
        raise ValueError('Each MODELS entry must be a (model_name, naming_convention) pair of nonempty strings.')
    naming_convention_spec(entry[1])
model_names = [name for name, _ in MODELS]
if len(model_names) != len(set(model_names)):
    raise ValueError('MODELS contains duplicate model IDs.')

run_results = {}
for model_name, naming_convention in MODELS:
    try:
        run_dir = cache_model(model_name, naming_convention)
        run_results[model_name] = {'status': 'complete', 'run_dir': str(run_dir)}
    except Exception as exc:
        run_results[model_name] = {'status': 'failed',
                                   'error': f'{type(exc).__name__}: {exc}',
                                   'traceback': traceback.format_exc()}
        print(run_results[model_name]['traceback'])
    finally:
        try:
            delete_local_model_data(model_name)
        except Exception as cleanup_error:
            cleanup_message = f'{type(cleanup_error).__name__}: {cleanup_error}'
            run_results.setdefault(model_name, {'status': 'failed'})['cleanup_error'] = cleanup_message
            print(f'WARNING: cleanup failed for {model_name}: {cleanup_message}')

display(run_results)
failures = {name: result for name, result in run_results.items() if result['status'] != 'complete'}
cleanup_failures = {name: result for name, result in run_results.items() if 'cleanup_error' in result}
if failures or cleanup_failures:
    raise RuntimeError(f'Caching failures: {list(failures)}; cleanup failures: {list(cleanup_failures)}')

## Next

Run `notebooks/2_bcpc_out_of_fold.ipynb`, then `notebooks/3_pls_arc_length_surface.ipynb`, against the same `artifacts/` tree, here or on any other machine. It needs no GPU, no model weights and no network: every model they find describes itself through its own `run_config.json`.

What to copy across, per model, is `artifacts/<model name>/`. The `.pt` activation caches dominate its size; keeping them is what lets the surface be refitted, and every inference corpus reprojected, without paying for this pass again.